# Pipeline to streamline the following tasks

 [1]. Extract YT video urls to create a curated dataset  
 >[VIDEO_ID, VIDEO_TITLE, VIDEO_URL, CHANNEL, DURATION_SECS,
UPLOAD_DATE, LANGUAGE, VIEW_COUNT, EXTRACTION_TIMESTAMP,
AUDIO_PATH, TRANSCRIPT_PATH, TRANSCRIPT_STATUS]  

 2. Extract audios from the videos using URLS or video_IDs.
 >Strip the .mp3 codec audio files from the videos and save them as a dataset.
 >> DataSet > VideoID > [audio.mp3]

 3. Establish process to extract / generate transcriptions from the videos.
 >Likely use of AI models for transcription due to no presence of youtube generated transcripts in the videos.

In [ ]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')

##  Module Installation

In [ ]:
%%capture
!pip install yt-dlp -U

## Module Imports

In [ ]:
import yt_dlp
import pandas as pd
import datetime

## Main Variables

In [ ]:
video_channel = []
video_IDs = []
video_durations = []
video_language = []
videos_LARGE = []
video_titles = []
upload_dates = []
view_counts = []
video_languages = []
extraction_timestamps = []
audio_paths = []
transcript_paths = []
transcript_statuses = []
video_language_INFO = {
    'Hindi'                                     : "hi",
    'Kashmiri'                                  : "ks",
    'Assamese'                                  : "as",
    "Marathi"                                   : "mr",
    "Gujarati"                                  : "gu",
    "Kannada"                                   : "kn",
    "Odia"                                      : "od",
    "Bengali"                                   : "bn"
}

### Constant for yt_dlp

In [ ]:
ydl_opts = {
                'extract_flat': True,
                'skip_download': True,
                'ignoreerrors': True,
            }

In [ ]:
#Helper Functions

def Larger_Video_CHECK(VIDEO_ID: str, DURATION: int):
    """Helper function to move larger videos to another area for processing later on"""
    if(DURATION > 1800):
        videos_LARGE.append(VIDEO_ID)
        return True

    return False


def YT_Link_Generator(VIDEO_ID :str) -> str:
    """Generator Function to generate YT_Link from VIDEO_ID"""
    return "https://www.youtube.com/watch?v=" + VIDEO_ID



## Get YT Channel Usernames and Playlists

In [ ]:
Sources = []

print("Please Provide the complete clean links")
def Get_Sources():
    while(1):
        print("Add the sources from which you want to source the YT Videos(Press 'N'/'n' to stop.):")
        a = input().strip()
        if(a in ['N','n']):
            break;

        if a=='': continue

        Sources.append(a)

Get_Sources()

### Function to fetch and extract video details to formulate the csv dataset

In [ ]:
### Start Video ID, duration, extraction

def extract_video_details(Video_URL : str):

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(Video_URL, download=False)

        for entry in info['entries']:
            video_channel.append(info.get('channel'))
            video_IDs.append(entry.get('id'))
            video_durations.append(entry.get('duration'))
            video_titles.append(entry.get('title'))
            upload_dates.append(entry.get('upload_date')) # YYYYMMDD string
            view_counts.append(entry.get('view_count'))
            # yt-dlp 'language' field can be inconsistent, use 'subtitles' if direct 'language' is insufficient
            video_languages.append(entry.get('language'))
            extraction_timestamps.append(datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
            audio_paths.append(None)
            transcript_paths.append(None)
            transcript_statuses.append(None)

%%capture
for source in Sources:
    extract_video_details(source)

### Form the .csv Dataset



In [ ]:
##Form csv dataset
#.csv dataset creation with [SrNo. , YT_VIDEO_ID, Duration, CHANNEL]
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "VIDEO_ID": video_IDs,
    "VIDEO_TITLE": video_titles,
    "VIDEO_LINK": [YT_Link_Generator(ID) for ID in video_IDs],
    "CHANNEL": video_channel,
    "DURATION_SECS": video_durations,
    "UPLOAD_DATE": upload_dates,
    "LANGUAGE": video_languages,
    "VIEW_COUNT": view_counts,
    "EXTRACTION_TIMESTAMP": extraction_timestamps,
    "AUDIO_PATH": audio_paths,
    "TRANSCRIPT_PATH": transcript_paths,
    "TRANSCRIPT_STATUS": transcript_statuses
})

df.to_csv("/content/drive/MyDrive/AnnamAI Tasks/YT_DATASET.csv", index = False)